# MoMo-FDVS logical PR15 — transaction training and calibration

This owner-operated notebook trains one source-specific `transaction_core` candidate from an exact PR14 bundle. It compares bounded baselines/candidates, calibrates independently, versions thresholds, verifies reload parity and exports private evidence. The locked-test partition is never loaded.

In [ ]:
RUN_PROFILE = "smoke"  # reproducibility preflight only
TRAINING_PROFILE = "full"
TARGET_COMMIT = "REPLACE_WITH_PUSHED_PR15_SHA"
REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/momo-fraud"
VM_ROOT = "/content/momo-work"
NOTEBOOK_PATH = "ml/notebooks/colab/04_train_transaction_models.ipynb"
DATASET_ID = "paysim"  # run paysim, momtsim-v1 and momtsim-v2 separately
MODEL_VERSION = f"transaction-core-{DATASET_ID}-pr15-v1"
FULL_ACKNOWLEDGEMENT = "I_ACKNOWLEDGE_FULL_COLAB_TRAINING"
assert RUN_PROFILE == "smoke"
assert TRAINING_PROFILE == "full"
assert DATASET_ID in {"paysim", "momtsim-v1", "momtsim-v2"}
assert len(TARGET_COMMIT) == 40 and TARGET_COMMIT != "REPLACE_WITH_PUSHED_PR15_SHA"

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive", force_remount=True, timeout_ms=600000)
repo = Path(VM_ROOT) / "repo"
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--prune", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", TARGET_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(repo / "ml/requirements-runtime.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(repo / "ml/requirements-training.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(repo / "ml")], check=True)
sys.path.insert(0, str(repo / "ml/src"))

In [ ]:
import json
from momo_fdvs_ml.colab import ColabPaths, colab_preflight_report
from momo_fdvs_ml.execution import ExecutionProfile, require_training_execution

paths = ColabPaths(drive_root=Path(DRIVE_ROOT), vm_root=Path(VM_ROOT))
preflight = colab_preflight_report(repo, paths=paths, profile=ExecutionProfile.SMOKE, notebook=NOTEBOOK_PATH, require_colab=True)
assert preflight["git"]["commit"] == TARGET_COMMIT
assert preflight["git"]["dirty"] is False
assert preflight["full_training_executed"] is False
require_training_execution(ExecutionProfile.FULL, acknowledgement=FULL_ACKNOWLEDGEMENT)
print(json.dumps({"commit": TARGET_COMMIT, "preflight_profile": RUN_PROFILE, "training_profile": TRAINING_PROFILE, "dataset_id": DATASET_ID, "locked_test_access_allowed": False}, indent=2, sort_keys=True))

In [ ]:
from momo_fdvs_ml.manifest import sha256_file
from momo_fdvs_ml.transaction_model import load_training_config, train_and_package_transaction_core

source_hashes = {
    "paysim": "f7eef9ffad5cfa64a034143a5c9b30491d189420b273d5ad5723ca40b596613d",
    "momtsim-v1": "da951eb95735da96271740a3e66b676b342d3831ce3111cd19dbfa020d3bd0a7",
    "momtsim-v2": "642fcb2ba7c9cbfffb933729d118f426fefddcbaabbf002793807be169fe80cd",
}
source_sha256 = source_hashes[DATASET_ID]
dataset_roots = {dataset_id: Path(DRIVE_ROOT) / "runs/pr14-transaction-features" / f"{dataset_id}-{digest[:12]}" for dataset_id, digest in source_hashes.items()}
dataset_root = dataset_roots[DATASET_ID]
external_dataset_roots = [root for dataset_id, root in dataset_roots.items() if dataset_id != DATASET_ID]
output_dir = Path(DRIVE_ROOT) / "runs/pr15-transaction-models" / MODEL_VERSION
config_path = repo / "ml/configs/transaction_core_default.json"
training_lock = repo / "ml/requirements-training.lock"
assert dataset_root.is_dir(), "The exact PR14 Drive bundle is missing."
outputs = train_and_package_transaction_core(dataset_root=dataset_root, output_dir=output_dir, model_version=MODEL_VERSION, training_commit_sha=TARGET_COMMIT, notebook=NOTEBOOK_PATH, dependency_lock_sha256=sha256_file(training_lock), config=load_training_config(config_path), external_dataset_roots=external_dataset_roots)
safe_summary = {
    "dataset_id": outputs.report["dataset_id"],
    "model_version": outputs.report["model_version"],
    "artifact_sha256": outputs.artifact_sha256,
    "split_manifest_sha256": outputs.report["split_manifest_sha256"],
    "preprocessor_sha256": outputs.report["preprocessor_sha256"],
    "config_sha256": outputs.report["config_sha256"],
    "selection": outputs.report["selection"],
    "calibration": outputs.report["calibration"],
    "thresholds": outputs.report["thresholds"],
    "external_tuning_evaluations": outputs.report["external_tuning_evaluations"],
    "locked_test_sealed": outputs.report["locked_test_sealed"],
    "locked_test_accessed_for_decisions": outputs.report["locked_test_accessed_for_decisions"],
    "final_evaluation_executed": outputs.report["final_evaluation_executed"],
    "not_real_world_probability": outputs.report["not_real_world_probability"],
    "promotable": outputs.report["promotable"],
}
print(json.dumps(safe_summary, indent=2, sort_keys=True))
assert outputs.report["locked_test_sealed"] is True
assert outputs.report["locked_test_accessed_for_decisions"] is False
assert outputs.report["final_evaluation_executed"] is False
assert outputs.report["not_real_world_probability"] is True
assert outputs.report["promotable"] is False

## Stop boundary

Run this notebook separately for PaySim, MoMTSim v1 and MoMTSim v2. Report tuning/calibration evidence only. Do not access any `locked_test` shard, perform final evaluation, activate a bundle, claim a real-world fraud probability or begin PR16 private screenshot collection until the PR15 evidence is reviewed.